# Ver un usuario por su id

## Qué hace

Muestra el usuario de ese id.

## Qué pasa en la base de datos

Solo lee. No cambia ninguna fila.

## Qué acción usa

`GET /users/:id`. Responde `200` y el usuario. La celda busca a Ada por su email, `ada@example.com`, para usar su id; si no la encuentra, usa `1`. Crea a Ada antes con `create-user`.

## Anotaciones

Si el id no es un entero mayor que cero, como `abc` o `1.5`, se ve `User id must be a positive integer`. Si ese usuario no existe, se ve `User does not exist`. Si no se llega a la base, se ve `Unable to find user`. Se abre con `pnpm run jupyter-node-http` desde la app, con el servidor de datos (`pnpm start`) y este cliente (`pnpm run node-http`) arrancados.


In [ ]:
import os
import subprocess

# The cell runs Node: Python only starts it and prints what it writes.
code = r"""
import Table from "cli-table3";

const origin = "http://127.0.0.1:3004";
let response;
try {
  // Look for Ada by her email to get her id. If the list fails or Ada
  // is not there, the id stays 1 and the answer says what happened.
  let id = 1;
  const listResponse = await fetch(origin + "/users");
  if (listResponse.ok) {
    const users = await listResponse.json();
    const ada = users.find((user) => user.email === "ada@example.com");
    if (ada) {
      id = ada.id;
    }
  }
  response = await fetch(origin + "/users/" + id);
} catch {
  console.log("Unable to reach the API");
  process.exit(0);
}

const text = await response.text();
let result = null;
if (text !== "") {
  result = JSON.parse(text);
}

if (response.ok === false) {
  // A failure shows its message and never the table of data.
  console.log(result.message);
} else if (result === null) {
  const table = new Table({
    head: ["Result"],
    style: { head: ["cyan", "bold"] },
  });
  table.push(["No data returned."]);
  console.log(table.toString());
} else {
  const table = new Table({
    head: ["id", "name", "email", "createdAt", "updatedAt"],
    style: { head: ["cyan", "bold"] },
  });
  let users = result;
  if (Array.isArray(result) === false) {
    users = [result];
  }
  for (const user of users) {
    table.push([
      user.id,
      user.name,
      user.email,
      user.createdAt,
      user.updatedAt,
    ]);
  }
  console.log(table.toString());
  if (users.length === 0) {
    console.log("No rows.");
  }
}
"""

# Colors need FORCE_COLOR because this output is not a terminal.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")
